# GEO-Bench-VLM — Qwen2-VL-7B-Instruct Evaluation

**Required GPU:** L4 (24 GB) or A100 (40 GB)  
**Prerequisite:** Run `notebooks/download_dataset.ipynb` once to download the dataset to Drive.  
Results are saved to Google Drive. Run cells top-to-bottom.

In [ ]:
# ── CONFIG — only edit this cell ─────────────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/GEOBench'  # root folder on your Drive
REPO_URL     = 'https://github.com/The-AI-Alliance/GEO-Bench-VLM'
MAX_SAMPLES  = None   # int (e.g. 10) for a smoke test, None for full benchmark

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ── 2. Mount Drive & configure paths ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

WEIGHTS_DIR = f'{DRIVE_BASE}/Out_weights'
DATA_PATH   = f'{DRIVE_BASE}/GEOBench-VLM'
REPO        = '/content/GEO-Bench-VLM'

for d in [WEIGHTS_DIR, DATA_PATH]:
    os.makedirs(d, exist_ok=True)

os.environ['DATA_PATH']                = DATA_PATH
os.environ['WEIGHTS_DIR']             = WEIGHTS_DIR
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'DATA_PATH   = {DATA_PATH}')
print(f'WEIGHTS_DIR = {WEIGHTS_DIR}')
print('Results will be saved inside DATA_PATH (already on Drive).')

In [ ]:
# ── 3. Clone repo ─────────────────────────────────────────────────────────────
import os
REPO = '/content/GEO-Bench-VLM'
if not os.path.isdir(REPO):
    !git clone {REPO_URL} {REPO}

weights_link = f'{REPO}/Out_weights'
if not os.path.exists(weights_link):
    os.symlink(os.environ['WEIGHTS_DIR'], weights_link)

%cd {REPO}
print('Repo ready.')

In [ ]:
# ── 4. HuggingFace login ──────────────────────────────────────────────────────
# Add HF_TOKEN to Colab Secrets (key icon in sidebar) for unattended runs.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in via Colab secret HF_TOKEN.')
except Exception:
    from huggingface_hub import login
    login()  # fallback: interactive prompt

In [ ]:
# ── 5. Verify dataset is present ─────────────────────────────────────────────
# Run notebooks/download_dataset.ipynb FIRST if you haven't already.
import os

dataset_dir = os.environ['DATA_PATH']
single_dir  = os.path.join(dataset_dir, 'Single')

if not os.path.isdir(single_dir):
    raise FileNotFoundError(
        f"Dataset not found at {dataset_dir}\n"
        "Please run notebooks/download_dataset.ipynb first."
    )

splits = ['Single', 'Temporal', 'Captioning', 'Ref-Det', 'Ref-Seg']
for split in splits:
    qa = os.path.join(dataset_dir, split, 'qa.json')
    status = '✓' if os.path.isfile(qa) else '✗ missing'
    print(f'  {split:<12} {status}')
print('Dataset ready.')

In [ ]:
# ── 6. Install dependencies ───────────────────────────────────────────────────
!pip install -q "transformers>=4.57.0" accelerate "qwen-vl-utils[decord]==0.0.14" \
    pillow pandas tqdm
print('Dependencies installed.')

In [ ]:
# ── 7. Download model weights (idempotent) ────────────────────────────────────
import os
from huggingface_hub import snapshot_download

model_dir = os.path.join(os.environ['WEIGHTS_DIR'], 'Qwen2-VL-7B-Instruct')
if os.path.isdir(model_dir):
    print('Weights already present — skipping download.')
else:
    print('Downloading Qwen2-VL-7B-Instruct weights...')
    snapshot_download('Qwen/Qwen2-VL-7B-Instruct', local_dir=model_dir)
    print('Weights downloaded.')

In [ ]:
# ── 8. Run evaluation ─────────────────────────────────────────────────────────
import subprocess, sys, os

REPO = '/content/GEO-Bench-VLM'
cmd  = [sys.executable, 'runmodel.py', 'qwen',
        '--data_path', os.environ['DATA_PATH']]
if MAX_SAMPLES:
    cmd += ['--max_samples', str(MAX_SAMPLES)]

result = subprocess.run(cmd, cwd=f'{REPO}/eval_geobenchvlm')
if result.returncode != 0:
    raise RuntimeError('Evaluation failed — see output above.')
print('Evaluation complete.')

In [ ]:
# ── 9. Confirm results saved to Drive ────────────────────────────────────────
# Results are written directly into DATA_PATH which lives on Drive — no copy needed.
import glob, os, json

results_folder = os.path.join(os.environ['DATA_PATH'], 'Results-qwen2-countcls')
json_files = glob.glob(os.path.join(results_folder, '*.json'))

if not json_files:
    print('No result files found — did the eval complete?')
else:
    print(f'Results saved at: {results_folder}')
    for f in json_files:
        with open(f) as fh:
            data = json.load(fh)
        print(f'  {os.path.basename(f)} — {len(data)} entries')
    print('\nRun notebooks/score_results.ipynb to compare models.')